In [1]:
import pandas as pd

players = pd.read_csv("data/raw/cookie_cats.csv", dtype="string")

print(f"Rows: {len(players):,}")
print("Columns:", players.columns.tolist())

Rows: 90,189
Columns: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


In [2]:
day_1 = players["retention_1"].str.lower()
day_7 = players["retention_7"].str.lower()

assert day_1.isin(["true", "false"]).all(), "Invalid day-1 flag"
assert day_7.isin(["true", "false"]).all(), "Invalid day-7 flag"

retention_counts = (
    players.assign(
        returned_day_1=day_1.eq("true"),
        returned_day_7=day_7.eq("true"),
    )
    .groupby("version", dropna=False)
    .agg(
        assigned_players=("userid", "size"),
        day_1_returns=("returned_day_1", "sum"),
        day_7_returns=("returned_day_7", "sum"),
    )
)

sql_counts = {
    "gate_30": {"assigned_players": 44700, "day_1_returns": 20034, "day_7_returns": 8502},
    "gate_40": {"assigned_players": 45489, "day_1_returns": 20119, "day_7_returns": 8279},
}

assert retention_counts.to_dict(orient="index") == sql_counts, "Counts differ from SQL"
retention_counts

,assigned_players,day_1_returns,day_7_returns
version,,,
gate_30,44700,20034,8502
gate_40,45489,20119,8279


In [3]:
from math import sqrt

assigned = retention_counts["assigned_players"]
day_7_rates = retention_counts["day_7_returns"] / assigned

rate_30 = day_7_rates.loc["gate_30"]
rate_40 = day_7_rates.loc["gate_40"]
day_7_difference = rate_40 - rate_30

standard_error = sqrt(
    rate_30 * (1 - rate_30) / assigned.loc["gate_30"]
    + rate_40 * (1 - rate_40) / assigned.loc["gate_40"]
)
ci_low = day_7_difference - 1.96 * standard_error
ci_high = day_7_difference + 1.96 * standard_error

print(f"Day-7 difference: {100 * day_7_difference:+.2f} pp")
print(f"95% confidence interval: [{100 * ci_low:+.2f}, {100 * ci_high:+.2f}] pp")

Day-7 difference: -0.82 pp
95% confidence interval: [-1.33, -0.31] pp


In [4]:
from math import erfc

expected_per_group = assigned.sum() / 2
allocation_chi_square = ((assigned - expected_per_group) ** 2 / expected_per_group).sum()

# Upper-tail probability for a chi-square statistic with one degree of freedom.
allocation_p_value = erfc(sqrt(allocation_chi_square / 2))

print(f"Expected per group under 50:50: {expected_per_group:,.1f}")
print(f"Chi-square: {allocation_chi_square:.3f}")
print(f"P-value: {allocation_p_value:.4f}")

Expected per group under 50:50: 45,094.5
Chi-square: 6.902
P-value: 0.0086


## Interpretation and decision

Day-7 retention was 19.02% with the level-30 gate and 18.20% with the level-40 gate.
The difference was −0.82 percentage points (95% CI: −1.33 to −0.31), roughly 8 fewer
day-7 players per 1,000 assigned to level 40. Day-1 retention was 44.82% versus 44.23%,
respectively, and serves as supporting context.

The source describes random assignment at install, so we compare all players in their assigned groups. If the intended allocation was 50:50, the observed split warrants investigation (p = 0.0086). The exact target is unverified, and the CSV lacks assignment logs and install timestamps needed to explain the imbalance or verify complete day-7 follow-up.

Keep the level-30 gate for now. Verify the planned split, assignment logs and follow-up
coverage before revisiting the decision. If those records are unavailable, rerun the
experiment with a documented split and agree in advance on the smallest retention change
that would justify moving the gate.

In [5]:
from importlib.metadata import version
from platform import python_version

print(f"Python: {python_version()}")
print(f"pandas: {pd.__version__}")
print(f"ipykernel: {version('ipykernel')}")

Python: 3.14.6
pandas: 3.0.3
ipykernel: 6.31.0
